In [2]:
!pip install scikeras

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Flatten, Input
from scikeras.wrappers import KerasClassifier

In [5]:
# Load preprocessed data (assumed to be cleansed, tokenized, stemmed, and labeled with IndoBERT)
data = pd.read_excel('FIX data_label 3 kls.xlsx')
data

,Unnamed: 0,created_at,full_text,username,clean,normal,token,stemmed,teks,label
0,1,2022-01-27 12:29:56+00:00,"gpp company lokal asal WFA, mungkin bukan peng...",xyberya,gpp company lokal asal wfa mungkin bukan penge...,tidak apa apa company lokal asal wfa mungkin b...,"['tidak', 'apa', 'apa', 'company', 'lokal', 'a...","['tidak', 'apa', 'apa', 'company', 'lokal', 'a...","[tidak, apa, apa, company, lokal, asal, wfa, m...",negative
1,2,2022-01-27 11:33:49+00:00,@arifsyh86 @pipis @MrsEuscha @tbputera Sorry b...,ninanenen,sorry baru liat notif tidak harus di jakarta k...,sorry baru liat notif tidak harus di jakarta k...,"['sorry', 'baru', 'liat', 'notif', 'tidak', 'h...","['sorry', 'baru', 'liat', 'notif', 'tidak', 'h...","[sorry, baru, liat, notif, tidak, harus, di, j...",negative
2,3,2022-01-27 09:40:40+00:00,Yay nambah opsi baru wfa spot di cibi ????????...,mirzarm,yay nambah opsi baru wfa spot di cibi,yay nambah opsi baru wfa spot di cibi,"['yay', 'nambah', 'opsi', 'baru', 'wfa', 'spot...","['yay', 'nambah', 'opsi', 'baru', 'wfa', 'spot...","[yay, nambah, opsi, baru, wfa, spot, di, cibi]",neutral
3,4,2022-01-27 04:39:49+00:00,Jadi pelaksana proyek susah kalo wfa. Apalagi ...,segedegabun,jadi pelaksana proyek susah kalo wfa apalagi a...,jadi pelaksana proyek susah kalau wfa apalagi ...,"['jadi', 'pelaksana', 'proyek', 'susah', 'kala...","['jadi', 'laksana', 'proyek', 'susah', 'kalau'...","[jadi, laksana, proyek, susah, kalau, wfa, apa...",negative
4,5,2022-01-27 00:30:58+00:00,tujuan karir sekarang fokusnya lebih ke work l...,favophs,tujuan karir sekarang fokusnya lebih ke work l...,tujuan karir sekarang fokusnya lebih ke work l...,"['tujuan', 'karir', 'sekarang', 'fokusnya', 'l...","['tuju', 'karir', 'sekarang', 'fokus', 'lebih'...","[tuju, karir, sekarang, fokus, lebih, ke, work...",positive
...,...,...,...,...,...,...,...,...,...,...
13644,13645,2022-09-02 01:34:53+00:00,cuuuy yg hari ini gabut dan mau work from anyw...,tsanianadh,cuuuy yg hari ini gabut dan mau work from anyw...,cuuuy yang hari ini menganggur dan mau work fr...,"['cuuuy', 'yang', 'hari', 'ini', 'menganggur',...","['cuuuy', 'yang', 'hari', 'ini', 'anggur', 'da...","[cuuuy, yang, hari, ini, anggur, dan, mau, wor...",positive
13645,13646,2022-09-01 15:34:44+00:00,"Ngambil toga rencana jalan dr kantor jam 4, ny...",mrcddsl,ngambil toga rencana jalan dr kantor jam nyamp...,ngambil toga rencana jalan dr kantor jam nyamp...,"['ngambil', 'toga', 'rencana', 'jalan', 'dr', ...","['ngambil', 'toga', 'rencana', 'jalan', 'dr', ...","[ngambil, toga, rencana, jalan, dr, kantor, ja...",neutral
13646,13647,2022-09-01 09:45:00+00:00,?6 Tips sukses interview online bagi remote wo...,remoteskillsac,tips sukses interview online bagi remote worke...,tips sukses interview online bagi remote worke...,"['tips', 'sukses', 'interview', 'online', 'bag...","['tips', 'sukses', 'interview', 'online', 'bag...","[tips, sukses, interview, online, bagi, remote...",neutral
13647,13648,2022-09-01 09:42:34+00:00,Udah gak jaman worfk from home work from offic...,mirzaftr,udah gak jaman worfk from home work from offic...,sudah tidak jaman worfk from home work from of...,"['sudah', 'tidak', 'jaman', 'worfk', 'from', '...","['sudah', 'tidak', 'jaman', 'worfk', 'from', '...","[sudah, tidak, jaman, worfk, from, home, work,...",negative


In [6]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(data['teks']).toarray()
y = data['label']

In [7]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
# Define CNN-LSTM model
def create_model(dropout_rate=0.5):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1], 1)))  # Input layer

    # CNN Layers
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))

    # LSTM Layers
    model.add(LSTM(64, activation='tanh', return_sequences=True))
    model.add(Dropout(dropout_rate))

    # Fully Connected Layers
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))  # Binary classification

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
from scikeras.wrappers import KerasClassifier

In [13]:
model = KerasClassifier(
    model=create_model,
    optimizer='adam',
    dropout_rate=0.5,
    epochs=10,
    batch_size=32,
    verbose=0
)

In [14]:
# GridSearch with 5-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(estimator=model, param_grid='param_grid', cv=cv, scoring='accuracy', verbose=1)
grid_result = grid.fit(X_train, y_train)

InvalidParameterError: The 'param_grid' parameter of GridSearchCV must be an instance of 'dict' or an instance of 'list'. Got 'param_grid' instead.

In [ ]:
# Best parameters and score
print(f"Best Parameters: {grid_result.best_params_}")
print(f"Best Accuracy: {grid_result.best_score_}")



In [ ]:
# Evaluate on test data
best_model = grid_result.best_estimator_
y_pred = best_model.predict(X_test)



In [ ]:
# Confusion Matrix and Classification Report
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))
